# 2.5 章节实践

搭建FastGelu算子的完整工程，理解算子的标准目录结构和Tiling实现。

要求：
- 按照标准目录结构创建FastGelu算子工程
- 理解op_host（算子定义、Tiling）、op_kernel（核函数）各层文件的职责
- 编写CMakeLists.txt并完成编译配置

In [ ]:
import os
import sys

# 创建章节工作目录
!mkdir -p Sources/02.05

# 加载昇腾环境变量
ascend_toolkit_home = os.environ.get("ASCEND_TOOLKIT_HOME", "/usr/local/Ascend/ascend-toolkit/latest")
print(f"ASCEND_TOOLKIT_HOME: {ascend_toolkit_home}")
!source {ascend_toolkit_home}/set_env.sh && echo "Environment loaded successfully."

---

## 1. 创建工程目录结构

FastGelu的标准算子工程目录如下，请补全缺失的目录和文件。

```
fast_gelu/
├── CMakeLists.txt              # 算子编译入口
├── op_host/                    # Host侧：算子定义、Tiling
│   ├── fast_gelu_def.cpp       # 算子定义
│   └── arch35/                 # ascend950架构Tiling
│       ├── fast_gelu_tiling_arch35.h
│       └── fast_gelu_tiling_arch35.cpp
├── op_kernel/                  # Device侧：Ascend C核函数
│   └── arch35/
├── op_api/                     # API层
├── op_graph/                   # 框架集成
├── examples/                   # 示例程序
└── tests/                      # 测试
    └── ut/
```

In [ ]:
!mkdir -p Sources/02.05/fast_gelu/op_host/arch35
!mkdir -p Sources/02.05/fast_gelu/op_kernel/arch35
!mkdir -p Sources/02.05/fast_gelu/op_api
!mkdir -p Sources/02.05/fast_gelu/op_graph
!mkdir -p Sources/02.05/fast_gelu/examples
!mkdir -p Sources/02.05/fast_gelu/tests/ut
print("Directory structure created.")

In [ ]:
# 查看创建的目录结构
!tree Sources/02.05/fast_gelu

---

## 2. 算子定义 (op_host/fast_gelu_def.cpp)

FastGelu的算子定义：输入 x (float)，输出 y (float)，绑定到 fast_gelu_kernel 和 fast_gelu_tiling。

In [ ]:
%%writefile Sources/02.05/fast_gelu/op_host/fast_gelu_def.cpp
#include "register/op_def_registry.h"
#include "tiling/tiling_data.h"

namespace ge {

class FastGeluOp : public OpDef {
public:
    explicit FastGeluOp(const char* name) : OpDef(name) {
        this->Input("x")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT})
            .Format({ge::FORMAT_ND});
        this->Output("y")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT})
            .Format({ge::FORMAT_ND});

        OpAICoreConfig cfg;
        cfg.ExtendCfgInfo("opFile.value", "fast_gelu_kernel");
        cfg.ExtendCfgInfo("tilingFile.value", "fast_gelu_tiling");
        this->AICore().AddConfig("ascend950", cfg);
    }
};

IMPL_OP_INFERSHAPE(FastGeluOp).InferShape(Ops::Base::InferShape4Elewise);
IMPL_OP_INFERSHAPE(FastGeluOp).InferDataType(Ops::Base::InferDtype4Elewise);

REG_OP(FastGelu)
    .Input("x", DT_FLOAT)
    .Output("y", DT_FLOAT)
    .OP_END();

}  // namespace ge


---

## 3. Tiling数据结构 (op_host/arch35/fast_gelu_tiling_arch35.h)

定义FastGeluTilingData结构体，包含totalLength和tileNum两个Tiling参数。

In [ ]:
%%writefile Sources/02.05/fast_gelu/op_host/arch35/fast_gelu_tiling_arch35.h
#ifndef FAST_GELU_TILING_ARCH35_H_
#define FAST_GELU_TILING_ARCH35_H_

#include "register/tilingdata_base.h"

namespace ge {

struct FastGeluTilingData {
    uint32_t totalLength;
    uint32_t tileNum;
};

}  // namespace ge

#endif  // FAST_GELU_TILING_ARCH35_H_


---

## 4. Tiling实现 (op_host/arch35/fast_gelu_tiling_arch35.cpp)

实现TilingFunc：从TilingContext获取输入形状，设置totalLength和tileNum=8，下发给Kernel。

In [ ]:
%%writefile Sources/02.05/fast_gelu/op_host/arch35/fast_gelu_tiling_arch35.cpp
#include "fast_gelu_tiling_arch35.h"
#include "register/op_impl_registry.h"
#include "tiling/tiling_api.h"

namespace ge {

IMPLEMT_VERIFIER(FastGelu, FastGeluVerify) {
    return GRAPH_SUCCESS;
}

IMPLEMT_COMMON_INIT_WITH_TILING_DATA_BEGIN(FastGelu)
    uint32_t totalLength = op_desc->GetInputDesc(0).GetShape().GetShapeSize();
    auto tilingData = reinterpret_cast<FastGeluTilingData*>(context->GetRawTilingData());
    tilingData->totalLength = totalLength;
    tilingData->tileNum = 8;
IMPLEMT_COMMON_INIT_WITH_TILING_DATA_END

}  // namespace ge


---

## 5. CMakeLists.txt编译配置

编写FastGelu的CMakeLists.txt，声明支持的芯片和编译源。

In [ ]:
%%writefile Sources/02.05/fast_gelu/CMakeLists.txt
set(SUPPORT_COMPUTE_UNIT "ascend950")
set(SUPPORT_TILING_DIR "arch35")
add_modules_sources(HOSTNAME ${OPHOST_NAME}
    MODE PRIVATE
    COMPUTE_UNIT ${SUPPORT_COMPUTE_UNIT}
    TILING_DIR ${SUPPORT_TILING_DIR}
    OPTYPE fast_gelu
    ACLNNTYPE aclnn_exclude)


---

## 6. 验证工程结构

查看完整的FastGelu算子工程目录结构。

In [ ]:
!tree Sources/02.05/fast_gelu/

---

## 答案参考

完成实践后，可以参考以下答案代码。

In [ ]:
!cat ./answer/02.05_answer/op_host/fast_gelu_def.cpp

In [ ]:
!cat ./answer/02.05_answer/op_host/arch35/fast_gelu_tiling_arch35.h

In [ ]:
!cat ./answer/02.05_answer/op_host/arch35/fast_gelu_tiling_arch35.cpp

In [ ]:
!cat ./answer/02.05_answer/CMakeLists.txt